<a href="https://colab.research.google.com/github/adi-bhagwat-v-s/Makemore/blob/main/Makemore_MLP_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
import random

In [2]:
from google.colab import files
uploaded = files.upload()

Saving names.txt to names.txt


In [3]:
# read all the words
words = open('names.txt', 'r').read().split()

In [4]:
# Create a mapping between words and integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [5]:
block_size = 3
def build_dataset(words):
  X, Y = [], []
  for w in words:
    context = [0]*block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [13]:
g = torch.Generator().manual_seed(2147483647)
n_emb = 10 # the dimensionality of the character embedding vector
n_hidden = 200 # Number of neuron in the hidden layer
vocab_size = 27 # 26 alphabets + '.'
C = torch.randn((vocab_size, n_emb), generator=g) # Lookup table
# Hidden Layer
W1 = torch.randn((block_size*n_emb, n_hidden), generator=g) * (5/3)/((block_size*n_emb)**0.5) # gain/fan-in
b1 = torch.rand(n_hidden, generator=g) * 0.01
'''tanh activation function converts extreme values to 1 or -1 and during back prop these gradients gets squashed to 0 resulting in dead neurons.
To prevent this the range of hpreact is reduced by scaling down W1 and b1'''
# Output Layer
W2 = torch.randn((n_hidden, vocab_size), generator=g)*0.01 # scaled down W2 will, in turn, reduce the logits(h@W2 + b1)
b2 = torch.randn(vocab_size, generator=g)*0
'''setting the bias to 0 helps prevent the overshooting of output values.
Maintaining the initial outputs close to 0 keeps the loss low at start.'''

# Batch normalization gain and bias
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]
for p in parameters:
  p.requires_grad = True

In [14]:
# @title
max_steps = 200000
batch_size = 32

for i in range(max_steps):
  # Minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  # Forward Pass
  emb = C[Xtr[ix]]
  embcat = emb.view(emb.shape[0], -1)

  # Linear Layer
  hpreact = embcat @ W1 # bias can be removed beacause in batch normalization the mean is subtracted from hpreact

  # Batch Normalisation
  bnmeani = hpreact.mean(0, keepdim=True)
  bnstdi = hpreact.std(0, keepdim=True)
  hpreact = bngain * ((hpreact - bnmeani) / bnstdi) + bnbias
  ''' Batch normalization - converting the output of a nn to a guassian format.'''
  with torch.no_grad():
    bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
    bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi

  h = torch.tanh(hpreact)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Ytr[ix])
  # Backward Pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # Update
  lr = 0.01 if i>=100000 else 0.1 # learning rate
  for p in parameters:
    p.data += -lr*p.grad

In [15]:
@torch.no_grad() # disables gradient tracking
def split_loss(split):
  x, y = {
      'train': (Xtr, Ytr),
      'val': (Xdev, Ydev),
      'test': (Xte, Yte)
  }[split]
  emb = C[x]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1
  #hpreact = bngain * (hpreact - hpreact.mean(0, keepdim=True)) / hpreact.std(0, keepdim=True) + bnbias
  hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
  h = torch.tanh(hpreact)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.06706166267395
val 2.1093485355377197


In [16]:
# Sample from the model
g = torch.Generator().manual_seed(2147483657)
for _ in range(20):
  out = []
  context = [0]*3 # 3 is the block size
  while True:
    emb = C[torch.tensor([context])]
    h = torch.tanh(emb.view(1, -1) @ W1 + b1)
    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
    context = context[1:] + [ix]
    out.append(ix)
    if ix==0:
      break

  print(''.join(itos[i] for i in out))

nathavflanghlrishwmlyx.
tetya.
kasshncjazonbbdavelystiagquifferlsistchrishbusthfy.
bhlton.
briqubbukssomigh.
vivliswatth.
gijarisixfatukcirslynestlciah.
gtngferstef.
prasmbusthmsyah.
mabosbasupcoll.
faldionennkelladlulus.
kesstton.
khrynesammalmanryah.
xennyxx.
fry.
bbdayvf.
qtevthell.
phristpr.
sillq.
kkmchryahqubigbrk.


In [10]:
logits = torch.tensor([0.0, 0.0, 0.0, 0.0])
probs = torch.softmax(logits, dim=0)
loss = -probs[0].log()
print("softmax => ", probs)
print("loss => ", loss);

softmax =>  tensor([0.2500, 0.2500, 0.2500, 0.2500])
loss =>  tensor(1.3863)
